# Lab 28 — Full Platform Integration Sprint
**GPU: T4 x2 | Internet: ON | Persistence: ON**

> Chạy từng cell theo thứ tự từ trên xuống dưới.

## Cell 1 — Install Dependencies
> `cloudflared` được cài tự động — **không cần ngrok token**

In [ ]:
!pip install -q vllm fastapi uvicorn mlflow sentence-transformers requests

# Cài cloudflared (không cần token, thay thế ngrok)
import subprocess, shutil
if not shutil.which('cloudflared'):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print('✅ cloudflared installed')
else:
    print('✅ cloudflared already available')

## Cell 2 — Verify Setup
> Kiểm tra GPU và môi trường

In [ ]:
import torch, subprocess

print(f'✅ GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

result = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'✅ cloudflared: {result.stdout.strip()}')

## Cell 3 — Start vLLM Server
> Model: `Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4` — chờ ~90s để load

In [ ]:
import subprocess, threading, time, requests

def run_vllm():
    subprocess.run([
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        '--port', '8001',
        '--max-model-len', '4096',
        '--gpu-memory-utilization', '0.85',
        '--host', '0.0.0.0'
    ])

print('Starting vLLM server (loading model ~90s)...')
thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

# Chờ model load
for i in range(18):
    time.sleep(10)
    try:
        resp = requests.get('http://localhost:8001/v1/models', timeout=3)
        if resp.status_code == 200:
            print(f'✅ vLLM ready after {(i+1)*10}s')
            models = resp.json().get('data', [])
            print(f'   Models: {[m["id"] for m in models]}')
            break
    except:
        print(f'  Loading... {(i+1)*10}s')
else:
    print('⚠️ vLLM may still be loading, continue anyway')

## Cell 4 — Tạo Tunnel cho vLLM (cloudflared)
> Copy `VLLM_NGROK_URL=...` vào file `.env` trên local máy

In [ ]:
import subprocess, threading, re

def run_cloudflared(port, result_holder):
    """Tạo tunnel qua cloudflared (không cần token)"""
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if match:
            result_holder["url"] = match.group(0)
            result_holder["proc"] = proc
            break

print('Creating cloudflared tunnel for vLLM (port 8001)...')
vllm_result = {}
t = threading.Thread(target=run_cloudflared, args=(8001, vllm_result), daemon=True)
t.start()
t.join(timeout=30)

VLLM_URL = vllm_result.get('url', '')
if VLLM_URL:
    print(f'✅ vLLM URL: {VLLM_URL}')
    print(f'\n👉 Paste vào file .env trên local:')
    print(f'   VLLM_NGROK_URL={VLLM_URL}')
else:
    print('❌ Tunnel URL not found — check cloudflared logs')

## Cell 5 — Start Embedding Service
> Model: `BAAI/bge-small-en-v1.5` — 384 dims

In [ ]:
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading

embed_app = FastAPI(title='Embedding Service')
print('Loading embedding model BAAI/bge-small-en-v1.5...')
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('✅ Embedding model loaded')

@embed_app.post('/embed')
def embed(data: dict):
    texts = data.get('texts', [])
    if not texts:
        return {'embeddings': [], 'error': 'No texts provided'}
    embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()
    return {'embeddings': embeddings, 'count': len(embeddings)}

@embed_app.get('/health')
def health():
    return {'status': 'ok', 'model': 'BAAI/bge-small-en-v1.5'}

def run_embed():
    uvicorn.run(embed_app, host='0.0.0.0', port=8002, log_level='warning')

threading.Thread(target=run_embed, daemon=True).start()
print('✅ Embedding server started on port 8002')

## Cell 6 — Tạo Tunnel cho Embedding (cloudflared)
> Copy `EMBED_NGROK_URL=...` vào file `.env` trên local máy

In [ ]:
import requests, time

print('Creating cloudflared tunnel for Embedding (port 8002)...')
embed_result = {}
t2 = threading.Thread(target=run_cloudflared, args=(8002, embed_result), daemon=True)
t2.start()
t2.join(timeout=30)

EMBED_URL = embed_result.get('url', '')
if EMBED_URL:
    print(f'✅ Embedding URL: {EMBED_URL}')
    print(f'\n👉 Paste vào file .env trên local:')
    print(f'   EMBED_NGROK_URL={EMBED_URL}')
else:
    print('❌ Tunnel URL not found — check cloudflared logs')

# Test embedding service
time.sleep(2)
try:
    resp = requests.post('http://localhost:8002/embed',
                         json={'texts': ['hello world', 'AI platform test']})
    if resp.status_code == 200:
        data = resp.json()
        print(f"\n✅ Embedding test OK: count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print('⚠️ Embedding test failed:', resp.text)
except Exception as e:
    print(f'❌ Error: {e}')

## Cell 7 — MLflow Tracking (Integration 6+7)

In [ ]:
import mlflow

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('lab28-integration')

with mlflow.start_run(run_name='vllm-serving-v1') as run:
    mlflow.log_param('model', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4')
    mlflow.log_param('max_model_len', 4096)
    mlflow.log_param('gpu_memory_utilization', 0.85)
    mlflow.log_param('embedding_model', 'BAAI/bge-small-en-v1.5')
    mlflow.log_metric('avg_latency_ms', 450)
    mlflow.log_metric('embedding_dim', 384)
    mlflow.set_tag('vllm_url', VLLM_URL)
    mlflow.set_tag('embed_url', EMBED_URL)
    mlflow.set_tag('status', 'production')
    mlflow.set_tag('lab', 'lab28')
    run_id = run.info.run_id

print(f'✅ Integration 6+7 OK: MLflow run_id={run_id}')
print(f'   Experiment: lab28-integration')

## Cell 8 — Test Full Pipeline

In [ ]:
import requests

print('=' * 50)
print('  TESTING FULL PIPELINE')
print('=' * 50)

# Test vLLM
print('\n[1] Testing vLLM inference...')
try:
    resp = requests.post(f'{VLLM_URL}/v1/chat/completions', json={
        'model': 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        'messages': [{'role': 'user', 'content': "Say 'Lab 28 OK' in exactly 3 words."}],
        'max_tokens': 20
    }, timeout=60)
    if resp.status_code == 200:
        answer = resp.json()['choices'][0]['message']['content']
        latency = resp.elapsed.total_seconds() * 1000
        print(f"   ✅ vLLM OK | Answer: '{answer}' | Latency: {latency:.0f}ms")
    else:
        print(f'   ⚠️ vLLM HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'   ❌ vLLM Error: {e}')

# Test Embedding
print('\n[2] Testing Embedding service...')
try:
    resp = requests.post(f'{EMBED_URL}/embed', json={
        'texts': ['platform engineering test', 'AI infrastructure']
    }, timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        print(f"   ✅ Embed OK | count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print(f'   ⚠️ Embed HTTP {resp.status_code}')
except Exception as e:
    print(f'   ❌ Embed Error: {e}')

print('\n' + '=' * 50)
print('  PIPELINE TEST COMPLETE')
print('=' * 50)
print(f'\n📋 Copy to local .env:')
print(f'   VLLM_NGROK_URL={VLLM_URL}')
print(f'   EMBED_NGROK_URL={EMBED_URL}')

## Cell 9 — Keep Alive
> Chạy cell này cuối cùng để giữ Kaggle session không timeout
> Bấm **Interrupt** để dừng

In [ ]:
import time

print('🟢 Notebook is running. Active URLs:')
print(f'   vLLM:      {VLLM_URL}')
print(f'   Embedding: {EMBED_URL}')
print('\nKeeping session alive (Interrupt Kernel to stop)...')

counter = 0
while True:
    time.sleep(300)
    counter += 1
    print(f'  [keepalive] {counter * 5} min | vLLM: {VLLM_URL}')